In [6]:
import random

In [8]:

# 1. Card Class

class Card:
    COLORS = ["Red", "Blue", "Green", "Yellow"]
    SPECIALS = ["Skip"]

    def __init__(self, color, value):
        self.color = color      # Red, Blue, Green, Yellow
        self.value = value      # 0-9 or Skip

    def is_special(self):
        return self.value == "Skip"

    def matches(self, other):
        #True if this card can be played on top of `other`.
        return self.color == other.color or self.value == other.value

    def __repr__(self):
        return f"{self.color} {self.value}"

    def __eq__(self, other):
        return isinstance(other, Card) and self.color == other.color and self.value == other.value

    def __hash__(self):
        return hash((self.color, self.value))



# 2. Deck Generator

def build_deck():
    deck = []
    for color in Card.COLORS:
        for num in range(10):          # 0-9
            deck.append(Card(color, num))
            deck.append(Card(color, num))  # two of each
        deck.append(Card(color, "Skip"))
        deck.append(Card(color, "Skip"))
    random.shuffle(deck)
    return deck



# 3. Legal Move Generator

def get_valid_moves(hand, top_card):
    """Return list of (index, card) tuples that are playable."""
    return [(i, card) for i, card in enumerate(hand) if card.matches(top_card)]


# ─────────────────────────────────────────────
# 4. State Representation
# ─────────────────────────────────────────────
def make_state(hands, top_card, deck, current_player=0, skip_next=False):
    return {
        "hands": [list(h) for h in hands],   # [p1_hand, p2_hand, p3_hand]
        "top_card": top_card,
        "deck": list(deck),
        "current_player": current_player,
        "skip_next": skip_next,
    }

def clone_state(state):
    return {
        "hands": [list(h) for h in state["hands"]],
        "top_card": state["top_card"],
        "deck": list(state["deck"]),
        "current_player": state["current_player"],
        "skip_next": state["skip_next"],
    }



# 5. State Transition

def apply_move(state, player_idx, move_card):
    """
    Apply a move. move_card=None means draw.
    Returns new state (cloned).
    """
    s = clone_state(state)
    hand = s["hands"][player_idx]

    if move_card is None:
        # Draw a card
        if s["deck"]:
            drawn = s["deck"].pop()
            hand.append(drawn)
    else:
        # Play the card
        hand.remove(move_card)
        s["top_card"] = move_card
        if move_card.is_special():
            s["skip_next"] = True

    # Advance turn
    next_p = (player_idx + 1) % 3
    if s["skip_next"] and move_card is not None and move_card.is_special():
        s["skip_next"] = False
        next_p = (player_idx + 2) % 3   # skip next player

    s["current_player"] = next_p
    return s


def is_terminal(state):
    for hand in state["hands"]:
        if len(hand) == 0:
            return True
    return False


def winner(state):
    for i, hand in enumerate(state["hands"]):
        if len(hand) == 0:
            return i
    return None



# 6. Evaluation Functions

def evaluate(state, player_idx, strategy="balanced"):
    """
    Score = 50 - 5*C_AI + 2*C_opp_avg + 3*S
    Weights are tuned per strategy.
    """
    hand = state["hands"][player_idx]
    opp_indices = [i for i in range(3) if i != player_idx]
    opp_cards = [len(state["hands"][i]) for i in opp_indices]
    c_ai = len(hand)
    c_opp = sum(opp_cards) / len(opp_indices)
    s = sum(1 for c in hand if c.is_special())

    if strategy == "defensive":
        # Heavier penalty for own cards, big reward for skips
        w1, w2, w3 = 6, 1.5, 5
    elif strategy == "offensive":
        # Reward shedding cards aggressively, less concern for skips
        w1, w2, w3 = 5, 3, 2
    else:
        w1, w2, w3 = 5, 2, 3

    return 50 - w1 * c_ai + w2 * c_opp + w3 * s



# 7. Minimax (P1 – Defensive, P3 – Simulation)

_tree_log = []   # collects tree nodes for display

def minimax(state, depth, maximizing_player_idx, current_depth=0, log_tree=False, tree_path="root"):
    if is_terminal(state) or depth == 0:
        score = evaluate(state, maximizing_player_idx, strategy="defensive")
        if log_tree:
            _tree_log.append({
                "path": tree_path,
                "depth": current_depth,
                "type": "LEAF",
                "score": score,
                "top_card": str(state["top_card"]),
                "hand": [str(c) for c in state["hands"][maximizing_player_idx]],
            })
        return score, None

    cp = state["current_player"]
    hand = state["hands"][cp]
    valid = get_valid_moves(hand, state["top_card"])
    moves = [card for _, card in valid] + [None]   # None = draw

    if cp == maximizing_player_idx:
        best_score = float("-inf")
        best_move = moves[0]
        for i, move in enumerate(moves):
            child = apply_move(state, cp, move)
            child_label = f"Play({move})" if move else "Draw"
            score, _ = minimax(child, depth - 1, maximizing_player_idx,
                               current_depth + 1, log_tree,
                               f"{tree_path} -> [{child_label}]")
            if log_tree:
                _tree_log.append({
                    "path": tree_path,
                    "depth": current_depth,
                    "type": "MAX",
                    "move": str(move) if move else "Draw",
                    "score": score,
                    "top_card": str(state["top_card"]),
                })
            if score > best_score:
                best_score = score
                best_move = move
        return best_score, best_move
    else:
        # Opponent: minimizes
        worst_score = float("inf")
        for i, move in enumerate(moves):
            child = apply_move(state, cp, move)
            child_label = f"Play({move})" if move else "Draw"
            score, _ = minimax(child, depth - 1, maximizing_player_idx,
                               current_depth + 1, log_tree,
                               f"{tree_path} -> [{child_label}]")
            if log_tree:
                _tree_log.append({
                    "path": tree_path,
                    "depth": current_depth,
                    "type": "MIN",
                    "move": str(move) if move else "Draw",
                    "score": score,
                    "top_card": str(state["top_card"]),
                })
            if score < worst_score:
                worst_score = score
        return worst_score, None



# 8. Expectimax (P2 – Offensive)

def expectimax(state, depth, ai_player_idx, current_depth=0, log_tree=False, tree_path="root"):
    if is_terminal(state) or depth == 0:
        score = evaluate(state, ai_player_idx, strategy="offensive")
        if log_tree:
            _tree_log.append({
                "path": tree_path,
                "depth": current_depth,
                "type": "LEAF",
                "score": round(score, 2),
                "top_card": str(state["top_card"]),
                "hand": [str(c) for c in state["hands"][ai_player_idx]],
            })
        return score, None

    cp = state["current_player"]
    hand = state["hands"][cp]
    valid = get_valid_moves(hand, state["top_card"])
    play_moves = [card for _, card in valid]

    if cp == ai_player_idx:
        # MAX node
        best_score = float("-inf")
        best_move = None

        # Evaluate play moves
        for move in play_moves:
            child = apply_move(state, cp, move)
            score, _ = expectimax(child, depth - 1, ai_player_idx,
                                  current_depth + 1, log_tree,
                                  f"{tree_path} -> [Play({move})]")
            if log_tree:
                _tree_log.append({
                    "path": tree_path,
                    "depth": current_depth,
                    "type": "MAX",
                    "move": str(move),
                    "score": round(score, 2),
                })
            if score > best_score:
                best_score = score
                best_move = move

        # Evaluate draw as CHANCE node
        deck = state["deck"]
        if deck:
            chance_score = _chance_node(state, cp, ai_player_idx, depth, current_depth, log_tree, tree_path)
            if log_tree:
                _tree_log.append({
                    "path": tree_path,
                    "depth": current_depth,
                    "type": "CHANCE_ROOT",
                    "move": "Draw",
                    "score": round(chance_score, 2),
                })
            if chance_score > best_score:
                best_score = chance_score
                best_move = None  # draw

        if best_move is None and not play_moves and not deck:
            best_score = evaluate(state, ai_player_idx, strategy="offensive")

        return best_score, best_move

    else:
        # Opponent node: random legal move (expectimax treats opponents as random)
        all_moves = play_moves + ([None] if not play_moves else [])
        if not all_moves:
            all_moves = [None]
        scores = []
        for move in all_moves:
            child = apply_move(state, cp, move)
            score, _ = expectimax(child, depth - 1, ai_player_idx,
                                  current_depth + 1, log_tree,
                                  f"{tree_path} -> [Opp({move})]")
            scores.append(score)
        avg = sum(scores) / len(scores) if scores else 0
        return avg, None


def _chance_node(state, player_idx, ai_player_idx, depth, current_depth, log_tree, tree_path):
    #Expected value of drawing a card from the deck.
    deck = state["deck"]
    if not deck:
        return evaluate(state, ai_player_idx, strategy="offensive")

    total = len(deck)
    unique_cards = {}
    for card in deck:
        key = (card.color, card.value)
        unique_cards[key] = unique_cards.get(key, 0) + 1

    expected = 0.0
    for (color, value), count in unique_cards.items():
        prob = count / total
        drawn = Card(color, value)
        child = clone_state(state)
        child["hands"][player_idx].append(drawn)
        child["deck"] = [c for c in child["deck"] if not (c.color == color and c.value == value and c == drawn)]
        # Remove just one instance
        removed = False
        new_deck = []
        for c in child["deck"]:
            if not removed and c.color == color and c.value == value:
                removed = True
            else:
                new_deck.append(c)
        child["deck"] = new_deck
        child["current_player"] = (player_idx + 1) % 3

        score, _ = expectimax(child, depth - 1, ai_player_idx,
                               current_depth + 1, log_tree,
                               f"{tree_path} -> [Chance({drawn})]")
        expected += prob * score

    return expected



# 9. Game Tree Printer

def print_tree_summary(log, max_entries=30):
    print("\n" + "="*60)
    print("SEARCH TREE LOG (sample)")
    print("="*60)
    for entry in log[:max_entries]:
        indent = "  " * entry["depth"]
        t = entry["type"]
        score = entry.get("score", "?")
        move = entry.get("move", "")
        path = entry.get("path", "")
        print(f"{indent}[{t}] move={move!s:<20} score={score!s:<8}  | {path[-60:]}")
    if len(log) > max_entries:
        print(f"  ... ({len(log) - max_entries} more nodes)")


# 10. Main Game Loop

PLAYER_NAMES = ["P1-Minimax(Defensive)", "P2-Expectimax(Offensive)", "P3"]

def play_game(p3_mode="simulation", verbose=True):
    
    #p3_mode: "simulation" or "manual"
    
    deck = build_deck()

    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop())

    top_card = deck.pop()
    while top_card.is_special():   # avoid skip as first card
        deck.insert(0, top_card)
        top_card = deck.pop()

    state = make_state(hands, top_card, deck, current_player=0)

    if verbose:
        print("="*50)
        print("       UNO GAME  _ADVERSARIAL SEARCH")
        print("="*50)
        print(f"Starting top card: {top_card}")
        for i, h in enumerate(hands):
            print(f"  {PLAYER_NAMES[i]}: {h}")
        print()

    turn = 0
    max_turns = 200   # safety

    while not is_terminal(state) and turn < max_turns:
        cp = state["current_player"]
        hand = state["hands"][cp]
        valid = get_valid_moves(hand, state["top_card"])

        if verbose:
            print(f"   Turn {turn+1} | {PLAYER_NAMES[cp]} ---")
            print(f"  Top card : {state['top_card']}")
            print(f"  Hand     : {hand}")

        move_card = None
        algo_used = ""

        if cp == 0:
            # P1 Minimax Defensive
            _tree_log.clear()
            score, move_card = minimax(state, depth=3, maximizing_player_idx=0,
                                       log_tree=(turn == 0))
            algo_used = f"Minimax  | eval={score:.1f}"

        elif cp == 1:
            # P2 Expectimax Offensive
            _tree_log.clear()
            score, move_card = expectimax(state, depth=3, ai_player_idx=1,
                                          log_tree=(turn == 1))
            algo_used = f"Expectimax | eval={score:.2f}"

        else:
            # P3
            if p3_mode == "manual":
                move_card = _manual_input(hand, state["top_card"])
                algo_used = "Manual"
            else:
                score, move_card = minimax(state, depth=3, maximizing_player_idx=2)
                algo_used = f"Minimax(sim) | eval={score:.1f}"

        # Print all valid moves with scores at depth 1 (like example output)
        if verbose:
            _print_move_analysis(state, cp, valid)

        action = f"plays {move_card}" if move_card else "draws a card"
        if verbose:
            print(f"  Decision : {action}  [{algo_used}]")
            if turn in (0, 1) and _tree_log:
                print_tree_summary(_tree_log, max_entries=15)
            print()

        state = apply_move(state, cp, move_card)
        turn += 1

    w = winner(state)
    if verbose:
        print("="*50)
        if w is not None:
            print(f"  WINNER: {PLAYER_NAMES[w]} in {turn} turns!")
        else:
            print("Game ended — max turns reached (draw).")
        print("Final hands:")
        for i, h in enumerate(state["hands"]):
            print(f"  {PLAYER_NAMES[i]}: {len(h)} cards  {h}")
        print("="*60)

    return w, turn, state


def _print_move_analysis(state, player_idx, valid_moves):
    #Print expected score for each possible move (depth-1 preview).
    print(f"  Move analysis (depth-1):")
    play_options = [card for _, card in valid_moves]

    # Score each play option
    for card in play_options:
        child = apply_move(state, player_idx, card)
        if player_idx == 1:
            sc = evaluate(child, player_idx, strategy="offensive")
        else:
            sc = evaluate(child, player_idx, strategy="defensive")
        print(f"    Play {str(card):<20} → score {sc:.1f}")

    # Score draw
    if state["deck"]:
        # Estimate as average over deck
        sc = evaluate(state, player_idx, "offensive" if player_idx == 1 else "defensive") - 5
        print(f"    Draw card            → score {sc:.1f} (estimated)")


def _manual_input(hand, top_card):
    print("  Your hand:")
    valid = get_valid_moves(hand, top_card)
    valid_cards = [card for _, card in valid]
    for i, card in enumerate(hand):
        mark = " ✓" if card in valid_cards else ""
        print(f"    [{i}] {card}{mark}")
    print(f"    [d] Draw a card")
    while True:
        choice = input("  Enter index or 'd': ").strip().lower()
        if choice == 'd':
            return None
        try:
            idx = int(choice)
            card = hand[idx]
            if card in valid_cards:
                return card
            else:
                print("  Invalid move. Try again.")
        except (ValueError, IndexError):
            print("  Bad input. Try again.")



# 11. Run Comparison

def run_comparison(n=20):
    """Run n simulated games and report win stats."""
    wins = {0: 0, 1: 0, 2: 0, None: 0}
    total_turns = []
    print(f"\nRunning {n} simulated games...\n")
    for i in range(n):
        w, turns, _ = play_game(p3_mode="simulation", verbose=False)
        wins[w] = wins.get(w, 0) + 1
        total_turns.append(turns)
        if (i+1) % 5 == 0:
            print(f"  {i+1}/{n} games done...")

    print("\n" + "="*50)
    print("COMPARISON RESULTS")
    print("="*50)
    for p in range(3):
        pct = wins[p] / n * 100
        print(f"  {PLAYER_NAMES[p]:<30} wins: {wins[p]:>3} / {n}  ({pct:.1f}%)")
    print(f"  Draw/timeout:                   {wins.get(None,0):>3} / {n}")
    print(f"  Avg turns per game: {sum(total_turns)/len(total_turns):.1f}")
    print("="*50)
    return wins


if __name__ == "__main__":
    print("Select mode:")
    print("  1. Single game — simulation (all AI)")
    print("  2. Single game — manual (you play as P3)")
    print("  3. Run 20 comparison games")
    choice = input("Enter 1/2/3: ").strip()

    if choice == "1":
        play_game(p3_mode="simulation", verbose=True)
    elif choice == "2":
        play_game(p3_mode="manual", verbose=True)
    elif choice == "3":
        run_comparison(20)
    else:
        print("Invalid choice.")

Select mode:
  1. Single game — simulation (all AI)
  2. Single game — manual (you play as P3)
  3. Run 20 comparison games


Enter 1/2/3:  1


       UNO GAME  _ADVERSARIAL SEARCH
Starting top card: Blue 7
  P1-Minimax(Defensive): [Green 3, Blue 8, Green 4, Blue 6, Yellow 9]
  P2-Expectimax(Offensive): [Green 3, Yellow 1, Red 3, Green 5, Blue 1]
  P3: [Blue 5, Yellow 3, Red Skip, Red 5, Blue 0]

   Turn 1 | P1-Minimax(Defensive) ---
  Top card : Blue 7
  Hand     : [Green 3, Blue 8, Green 4, Blue 6, Yellow 9]
  Move analysis (depth-1):
    Play Blue 8               → score 33.5
    Play Blue 6               → score 33.5
    Draw card            → score 22.5 (estimated)
  Decision : plays Blue 8  [Minimax  | eval=32.0]

SEARCH TREE LOG (sample)
      [LEAF] move=                     score=32.0      | root -> [Play(Blue 8)] -> [Play(Blue 1)] -> [Play(Blue 5)]
    [MIN] move=Blue 5               score=32.0      | root -> [Play(Blue 8)] -> [Play(Blue 1)]
      [LEAF] move=                     score=32.0      | root -> [Play(Blue 8)] -> [Play(Blue 1)] -> [Play(Blue 0)]
    [MIN] move=Blue 0               score=32.0      | root -> 

In [ ]:
#1. Evaluation Function Explanation:
#The Evaluation Function is the core heuristic used by our AI agents to estimate the "value" or "desirability"of a specific game state. Since the 
#state space of UNO is too large to search to the terminal (win/loss) node within a reasonable time, the AI explores to a Depth of 3 and then
#applies this mathematical formula to the leaf nodes.
#The Mathematical Formula:
#The baseline scoring provided for the search algorithms is
  #  :Score = 50 - w1(C_AI) + w2(C_opp) + w3(S)
#Where:
#CAI: Number of cards in the current player's hand.
#Copp: Average cards held by the two opponents.
#S: Number of Special (Skip) cards currently in the player's hand.
 #   2. Strategy Tuning: Defensive vs. OffensiveTo 
#fulfill the project requirements for distinct behaviors, we tune the weights (w_1, w_2, w_3) differently for each AI agent:
#  Player 1: Defensive Strategy (Minimax)Weights: w1=6, w2=1.5, w3=5
 # Logic: P1 prioritizes Control. By assigning a high weight to w_3, the AI values holding onto "Skip" cards. It treats them as defensive tools to be played only when an opponent's card count (C_opp) becomes dangerously low, effectively blocking them from winning.
 #Player 2: Offensive Strategy (Expectimax)
 #Weights: w_1=5, w_2=3, w_3=2
 #Logic: P2 prioritizes Shedding. It focuses more on reducing its own card count and cares less about the utility of Special cards. It treats the "Draw" action as a Chance Node, calculating if the probability of drawing a playable card is worth more than the penalty of increasing C_AI
    

In [ ]:
#Concluding Section: Performance Analysis
#Best Performer: Usually Player 2 (Expectimax).

#Reasoning: In a game like UNO, which involves significant hidden information and randomness (the deck), Minimax often becomes "too paranoid."
#It assumes opponents will always have the perfect card to stop it. Expectimax, by calculating the Expected Value of the deck, takes calculated risks
#that allow it to shed cards faster, which is the primary path to victory in UNO.
